In [ ]:
from pathlib import Path
import sys

from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/pratik0620/adaptive-federated-aggregation.git"
REPO_ROOT = Path("/content/adaptive-federated-aggregation")
if not REPO_ROOT.exists():
    !git clone {REPO_URL} {REPO_ROOT}

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SRC_DIR = REPO_ROOT / "src"
if not SRC_DIR.exists():
    raise FileNotFoundError(f"src directory not found at {SRC_DIR}")

print("Repository:", REPO_ROOT)
print("Source modules:", SRC_DIR)

In [ ]:
# Install the packages required by this notebook and the shared preprocessing utilities.
%pip install -q numpy pandas scikit-learn matplotlib jupyter


## Configuration — Google Drive Paths

The next code cell defines the only machine-specific setting, `BASE_DRIVE_PATH`, and validates the three NIH metadata files. This notebook does not require an image directory and does not download or copy X-ray images.

# NIH ChestX-ray14 preprocessing

This notebook creates the NIH federated client as metadata-only CSV files. It keeps clean target positives and explicit `No Finding` negatives, excludes ambiguous label combinations, preserves patient IDs, and splits whole patients. Images are never copied or downloaded in this phase.

**Storage model:** Code and notebooks are stored in GitHub, while the large NIH metadata files and future image files are stored in Google Drive. The actual X-ray images will be downloaded later, after the target disease and required sample size are finalized.

In [ ]:
import json, platform
import numpy as np
import pandas as pd

from src.config import TARGET_DISEASE, RANDOM_SEED, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES
from src.dataset_utils import nih_clean_binary_labels, standardize_nih
from src.preprocessing_utils import (assert_no_leakage, assert_no_study_leakage, build_summary,
                                     require_file, sample_clean_cases, save_splits,
                                     save_summary, set_reproducibility,
                                     split_train_val_by_patient)

set_reproducibility(RANDOM_SEED)
print('Python:', platform.python_version())
print('pandas:', pd.__version__, 'numpy:', np.__version__)

# Google Drive configuration for raw NIH metadata and metadata-only outputs.
BASE_DRIVE_PATH = Path("/content/drive/MyDrive/Healthcare_FL")
NIH_ROOT = BASE_DRIVE_PATH / "raw" / "NIH"
NIH_METADATA_DIR = NIH_ROOT / "metadata"

METADATA_FILE = NIH_METADATA_DIR / "Data_Entry_2017.csv"
TRAIN_VAL_FILE = NIH_METADATA_DIR / "train_val_list.txt"
TEST_FILE = NIH_METADATA_DIR / "test_list.txt"
OUTPUT_DIR = BASE_DRIVE_PATH / "processed" / "client_01_NIH"

# Project-level disease names stay stable; each source can map them to its own label vocabulary.
NIH_LABEL_MAP = {"Pleural Effusion": "Effusion"}
NIH_TARGET_LABEL = NIH_LABEL_MAP.get(TARGET_DISEASE, TARGET_DISEASE)

required_files = [METADATA_FILE, TRAIN_VAL_FILE, TEST_FILE]
for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Required NIH dataset file not found: {file_path}")

metadata_path = require_file(METADATA_FILE, "NIH metadata CSV")
train_val_path = require_file(TRAIN_VAL_FILE, "NIH train/validation split list")
test_path = require_file(TEST_FILE, "NIH test split list")

print("Repository:", REPO_ROOT)
print("Google Drive:", BASE_DRIVE_PATH)
print("Dataset:", NIH_ROOT)
print("Metadata:", METADATA_FILE)
print("NIH train/validation split path:", TRAIN_VAL_FILE)
print("NIH test split path:", TEST_FILE)
print("Project target disease:", TARGET_DISEASE)
print("NIH source target label:", NIH_TARGET_LABEL)
print("All required NIH metadata files exist.")

## Load and inspect
NIH `Finding Labels` is pipe-separated. The clean binary policy is deliberately conservative: a target-containing study is positive; exactly `No Finding` is negative; every other combination is excluded.

In [ ]:
nih = pd.read_csv(metadata_path)
train_val_ids = pd.read_csv(train_val_path, header=None, names=['image_id'])
test_ids = pd.read_csv(test_path, header=None, names=['image_id'])
train_val_ids['image_id'] = train_val_ids['image_id'].astype(str).str.strip()
test_ids['image_id'] = test_ids['image_id'].astype(str).str.strip()

print('images:', len(nih))
print('unique patients:', nih['Patient ID'].nunique())
print('columns:', nih.columns.tolist())
print('official train/validation image IDs:', len(train_val_ids))
print('official test image IDs:', len(test_ids))
print('official split image overlap:', len(set(train_val_ids['image_id']) & set(test_ids['image_id'])))
display(nih.isna().sum().sort_values(ascending=False).head(20).to_frame('missing'))
display(nih['Finding Labels'].value_counts(dropna=False).head(20).to_frame('label_count'))

# NIH calls the project-level Pleural Effusion target simply Effusion.
clean, counts = nih_clean_binary_labels(
    nih,
    TARGET_DISEASE,
    source_target_label=NIH_TARGET_LABEL,
)
print('Counts before filtering:', counts)
print('Clean rows:', len(clean))

In [ ]:
# No image directory is required for this metadata-only phase.
standardized = standardize_nih(clean, image_dir=None)
standardized['image_id'] = standardized['image_id'].astype(str).str.strip()

official_train_val_image_ids = set(train_val_ids['image_id'])
official_test_image_ids = set(test_ids['image_id'])
if official_train_val_image_ids & official_test_image_ids:
    raise AssertionError('Official NIH train/validation and test lists overlap.')

official_train_val = standardized[standardized['image_id'].isin(official_train_val_image_ids)].copy()
official_test = standardized[standardized['image_id'].isin(official_test_image_ids)].copy()
missing_from_official_lists = set(standardized['image_id']) - (official_train_val_image_ids | official_test_image_ids)
print('Clean images missing from official lists:', len(missing_from_official_lists))

# Sampling is applied only to the official train/validation portion so the official test set is preserved.
official_train_val = sample_clean_cases(
    official_train_val,
    MAX_POSITIVE_SAMPLES,
    MAX_NEGATIVE_SAMPLES,
    RANDOM_SEED,
)
train_val_splits = split_train_val_by_patient(official_train_val, seed=RANDOM_SEED, val_size=0.1)
splits = {
    'train': train_val_splits['train'],
    'val': train_val_splits['val'],
    'test': official_test,
}

assert_no_leakage(splits)
assert_no_study_leakage(splits)
print('patient leakage checks: passed')
print('study leakage checks: passed where study_id exists')

print('positive count:', counts['positive_before_filter'])
print('negative count:', counts['negative_before_filter'])
print('excluded count:', counts['excluded_ambiguous_before_filter'])
print('official split usage status: True')
for split_name, split_frame in splits.items():
    print(
        f'{split_name} image count:', len(split_frame),
        '| positive:', int((split_frame['target'] == 1).sum()),
        '| negative:', int((split_frame['target'] == 0).sum()),
        '| unique patients:', split_frame['patient_id'].nunique(),
    )

display(pd.DataFrame([
    {
        'split': split_name,
        'images': len(split_frame),
        'positive': int((split_frame['target'] == 1).sum()),
        'negative': int((split_frame['target'] == 0).sum()),
        'unique_patients': split_frame['patient_id'].nunique(),
    }
    for split_name, split_frame in splits.items()
]))
display(splits['train'].head())

save_splits(splits, OUTPUT_DIR)
summary = build_summary(
    splits,
    'NIH',
    {
        'target_disease': TARGET_DISEASE,
        'source_target_label': NIH_TARGET_LABEL,
        'random_seed': RANDOM_SEED,
        'max_positive_samples_train_val': MAX_POSITIVE_SAMPLES,
        'max_negative_samples_train_val': MAX_NEGATIVE_SAMPLES,
        'filter_counts_before_filtering': counts,
        'official_split_lists_loaded': True,
        'official_split_used': True,
        'official_test_preserved': True,
        'image_paths_materialized': False,
    },
)
save_summary(summary, OUTPUT_DIR)
print(json.dumps(summary, indent=2))
print('Saved:', sorted(path.name for path in OUTPUT_DIR.glob('*')))